# Overall Verification System (Face Matching & Anti-Spoofing)

This notebook implements an integrated, multi-modal facial authentication system combining:
1. **Face Verification**: Matches input face embeddings against registered embeddings in the `embeddings/` directory using **MTCNN** and **InceptionResNetV1** (`pretrained='vggface2'`).
2. **Anti-Spoofing Detection**: Passive liveness detection using an ensemble of **MiniFASNetV2** (scale 2.7) and **MiniFASNetV1SE** (scale 4.0) enhanced with **Facial Landmark Rotation Normalization**, **Close-Distance Anomaly Guard**, and **Temporal Multi-Frame Smoothing** (7-frame rolling sample).

### Key Protections Added:
- **Rotation Alignment**: Uses 5-point facial landmarks to calculate roll rotation angle $\theta$ and warp rotated phone screens to an upright $0^\circ$ orientation before MiniFASNet evaluation. Extreme unnatural tilts ($|\theta| > 30^\circ$) are immediately flagged as spoof attacks.
- **Close-Range Distance Check**: Detects when a phone screen is held unnaturally close to the lens (filling $>38\%$ frame area or clipping scale expansion boundaries) and flags it as a distance anomaly attack.
- **Temporal Aggregation**: Averages liveness probabilities across a 7-frame temporal burst to eliminate single-frame screen glare and refresh rate flicker.

### Workflow:
1. Camera opens in a live feed window, asking the user to press `'c'` (or `'C'`) to capture a picture (or `'q'` to quit).
2. The captured frame sequence (7 consecutive temporal frames) passes through:
   - `verify_face(frames)` -> Checks face similarity against registered embeddings in `embeddings/`.
   - `anti_spoofing_detection(frames)` -> Evaluates normalized multi-frame temporal liveness.
3. The main function `overall_verification()` returns **`True` if and only if both functions return `True`**, otherwise **`False`**.

## 1. Environment & GPU Configuration

In [68]:
# Environment Information & Setup
import os
import sys
import time
import urllib.request
from collections import deque
import numpy as np
import cv2
import PIL
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from facenet_pytorch import MTCNN, InceptionResnetV1

print("=== Environment Information ===")
print(f"Python Version:     {sys.version.split()[0]}")
print(f"PyTorch Version:    {torch.__version__}")
print(f"Torchvision:        {torchvision.__version__}")
print(f"OpenCV Version:     {cv2.__version__}")
print(f"NumPy Version:      {np.__version__}")
print(f"Pillow Version:     {PIL.__version__}")
print(f"CUDA Available:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device Name:   {torch.cuda.get_device_name(0)}")
print("===============================")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Selected Compute Device: {device}")

EMBEDDINGS_DIR = "embeddings"
if not os.path.exists(EMBEDDINGS_DIR):
    os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
    print(f"Created '{EMBEDDINGS_DIR}' directory.")
else:
    print(f"Found '{EMBEDDINGS_DIR}' directory containing: {os.listdir(EMBEDDINGS_DIR)}")

=== Environment Information ===
Python Version:     3.12.3
PyTorch Version:    2.13.0+cu130
Torchvision:        0.28.0+cu130
OpenCV Version:     5.0.0
NumPy Version:      2.5.2
Pillow Version:     12.3.0
CUDA Available:     True
CUDA Device Name:   NVIDIA GeForce GTX 1650
Selected Compute Device: cuda
Found 'embeddings' directory containing: ['antik.npy', 'shrey_new.npy', 'shrey_photo.npy']


## 2. Face Verification Models Initialization (MTCNN & InceptionResNetV1)

In [69]:
# Load MTCNN face detector & InceptionResNetV1 feature extractor
print("Initializing MTCNN and InceptionResNetV1 models...")
mtcnn = MTCNN(image_size=160, margin=0, keep_all=False, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)
print("Face Verification models initialized and set to eval mode.")

Initializing MTCNN and InceptionResNetV1 models...
Face Verification models initialized and set to eval mode.


## 3. MiniFASNet Anti-Spoofing Architecture Definitions & Loading

PyTorch model definitions for `MiniFASNetV2` (scale 2.7) and `MiniFASNetV1SE` (scale 4.0).

In [70]:
class Conv_block(nn.Module):
    def __init__(self, in_c, out_c, kernel=(1, 1), stride=(1, 1), padding=(0, 0), groups=1):
        super(Conv_block, self).__init__()
        self.conv = nn.Conv2d(in_c, out_c, kernel_size=kernel, groups=groups,
                              stride=stride, padding=padding, bias=False)
        self.bn = nn.BatchNorm2d(out_c)
        self.prelu = nn.PReLU(out_c)

    def forward(self, x):
        return self.prelu(self.bn(self.conv(x)))

class Linear_block(nn.Module):
    def __init__(self, in_c, out_c, kernel=(1, 1), stride=(1, 1), padding=(0, 0), groups=1):
        super(Linear_block, self).__init__()
        self.conv = nn.Conv2d(in_c, out_channels=out_c, kernel_size=kernel,
                              groups=groups, stride=stride, padding=padding, bias=False)
        self.bn = nn.BatchNorm2d(out_c)

    def forward(self, x):
        return self.bn(self.conv(x))

class Depth_Wise(nn.Module):
    def __init__(self, c1, c2, c3, residual=False, kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=1):
        super(Depth_Wise, self).__init__()
        c1_in, c1_out = c1; c2_in, c2_out = c2; c3_in, c3_out = c3
        self.conv = Conv_block(c1_in, out_c=c1_out, kernel=(1, 1), padding=(0, 0), stride=(1, 1))
        self.conv_dw = Conv_block(c2_in, c2_out, groups=c2_in, kernel=kernel, padding=padding, stride=stride)
        self.project = Linear_block(c3_in, c3_out, kernel=(1, 1), padding=(0, 0), stride=(1, 1))
        self.residual = residual

    def forward(self, x):
        if self.residual:
            short_cut = x
        x = self.project(self.conv_dw(self.conv(x)))
        return (short_cut + x) if self.residual else x

class Residual(nn.Module):
    def __init__(self, c1, c2, c3, num_block, groups, kernel=(3, 3), stride=(1, 1), padding=(1, 1)):
        super(Residual, self).__init__()
        modules = [Depth_Wise(c1[i], c2[i], c3[i], residual=True, kernel=kernel, padding=padding, stride=stride, groups=groups) for i in range(num_block)]
        self.model = nn.Sequential(*modules)

    def forward(self, x):
        return self.model(x)

class SEModule(nn.Module):
    def __init__(self, channels, reduction=4):
        super(SEModule, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, channels // reduction, kernel_size=1, padding=0, bias=False)
        self.bn1 = nn.BatchNorm2d(channels // reduction)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Conv2d(channels // reduction, channels, kernel_size=1, padding=0, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        module_input = x
        x = self.sigmoid(self.bn2(self.fc2(self.relu(self.bn1(self.fc1(self.avg_pool(x)))))))
        return module_input * x

class Depth_Wise_SE(nn.Module):
    def __init__(self, c1, c2, c3, residual=False, kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=1, se_reduct=8):
        super(Depth_Wise_SE, self).__init__()
        c1_in, c1_out = c1; c2_in, c2_out = c2; c3_in, c3_out = c3
        self.conv = Conv_block(c1_in, out_c=c1_out, kernel=(1, 1), padding=(0, 0), stride=(1, 1))
        self.conv_dw = Conv_block(c2_in, c2_out, groups=c2_in, kernel=kernel, padding=padding, stride=stride)
        self.project = Linear_block(c3_in, c3_out, kernel=(1, 1), padding=(0, 0), stride=(1, 1))
        self.residual = residual
        self.se_module = SEModule(c3_out, se_reduct)

    def forward(self, x):
        if self.residual:
            short_cut = x
        x = self.project(self.conv_dw(self.conv(x)))
        if self.residual:
            x = self.se_module(x)
            return short_cut + x
        return x

class ResidualSE(nn.Module):
    def __init__(self, c1, c2, c3, num_block, groups, kernel=(3, 3), stride=(1, 1), padding=(1, 1), se_reduct=4):
        super(ResidualSE, self).__init__()
        modules = []
        for i in range(num_block):
            if i == num_block - 1:
                modules.append(Depth_Wise_SE(c1[i], c2[i], c3[i], residual=True, kernel=kernel, padding=padding, stride=stride, groups=groups, se_reduct=se_reduct))
            else:
                modules.append(Depth_Wise(c1[i], c2[i], c3[i], residual=True, kernel=kernel, padding=padding, stride=stride, groups=groups))
        self.model = nn.Sequential(*modules)

    def forward(self, x):
        return self.model(x)

class MiniFASNet(nn.Module):
    def __init__(self, keep, embedding_size=128, conv6_kernel=(5, 5), drop_p=0.2, num_classes=3, img_channel=3):
        super(MiniFASNet, self).__init__()
        self.embedding_size = embedding_size
        self.conv1 = Conv_block(img_channel, keep[0], kernel=(3, 3), stride=(2, 2), padding=(1, 1))
        self.conv2_dw = Conv_block(keep[0], keep[1], kernel=(3, 3), stride=(1, 1), padding=(1, 1), groups=keep[1])

        c1 = [(keep[1], keep[2])]; c2 = [(keep[2], keep[3])]; c3 = [(keep[3], keep[4])]
        self.conv_23 = Depth_Wise(c1[0], c2[0], c3[0], kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=keep[3])

        c1 = [(keep[4], keep[5]), (keep[7], keep[8]), (keep[10], keep[11]), (keep[13], keep[14])]
        c2 = [(keep[5], keep[6]), (keep[8], keep[9]), (keep[11], keep[12]), (keep[14], keep[15])]
        c3 = [(keep[6], keep[7]), (keep[9], keep[10]), (keep[12], keep[13]), (keep[15], keep[16])]
        self.conv_3 = Residual(c1, c2, c3, num_block=4, groups=keep[4], kernel=(3, 3), stride=(1, 1), padding=(1, 1))

        c1 = [(keep[16], keep[17])]; c2 = [(keep[17], keep[18])]; c3 = [(keep[18], keep[19])]
        self.conv_34 = Depth_Wise(c1[0], c2[0], c3[0], kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=keep[19])

        c1 = [(keep[19], keep[20]), (keep[22], keep[23]), (keep[25], keep[26]), (keep[28], keep[29]), (keep[31], keep[32]), (keep[34], keep[35])]
        c2 = [(keep[20], keep[21]), (keep[23], keep[24]), (keep[26], keep[27]), (keep[29], keep[30]), (keep[32], keep[33]), (keep[35], keep[36])]
        c3 = [(keep[21], keep[22]), (keep[24], keep[25]), (keep[27], keep[28]), (keep[30], keep[31]), (keep[33], keep[34]), (keep[36], keep[37])]
        self.conv_4 = Residual(c1, c2, c3, num_block=6, groups=keep[19], kernel=(3, 3), stride=(1, 1), padding=(1, 1))

        c1 = [(keep[37], keep[38])]; c2 = [(keep[38], keep[39])]; c3 = [(keep[39], keep[40])]
        self.conv_45 = Depth_Wise(c1[0], c2[0], c3[0], kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=keep[40])

        c1 = [(keep[40], keep[41]), (keep[43], keep[44])]; c2 = [(keep[41], keep[42]), (keep[44], keep[45])]; c3 = [(keep[42], keep[43]), (keep[45], keep[46])]
        self.conv_5 = Residual(c1, c2, c3, num_block=2, groups=keep[40], kernel=(3, 3), stride=(1, 1), padding=(1, 1))

        self.conv_6_sep = Conv_block(keep[46], keep[47], kernel=(1, 1), stride=(1, 1), padding=(0, 0))
        self.conv_6_dw = Linear_block(keep[47], keep[48], groups=keep[48], kernel=conv6_kernel, stride=(1, 1), padding=(0, 0))
        self.conv_6_flatten = nn.Flatten()
        self.linear = nn.Linear(512, embedding_size, bias=False)
        self.bn = nn.BatchNorm1d(embedding_size)
        self.drop = nn.Dropout(p=drop_p)
        self.prob = nn.Linear(embedding_size, num_classes, bias=False)

    def forward(self, x):
        out = self.conv_6_dw(self.conv_6_sep(self.conv_5(self.conv_45(self.conv_4(self.conv_34(self.conv_3(self.conv_23(self.conv2_dw(self.conv1(x))))))))))
        out = self.conv_6_flatten(out)
        if self.embedding_size != 512:
            out = self.linear(out)
        out = self.bn(out)
        out = self.drop(out)
        return self.prob(out)

class MiniFASNetSE(MiniFASNet):
    def __init__(self, keep, embedding_size=128, conv6_kernel=(5, 5), drop_p=0.75, num_classes=3, img_channel=3):
        super(MiniFASNetSE, self).__init__(keep=keep, embedding_size=embedding_size, conv6_kernel=conv6_kernel,
                                           drop_p=drop_p, num_classes=num_classes, img_channel=img_channel)
        c1 = [(keep[4], keep[5]), (keep[7], keep[8]), (keep[10], keep[11]), (keep[13], keep[14])]
        c2 = [(keep[5], keep[6]), (keep[8], keep[9]), (keep[11], keep[12]), (keep[14], keep[15])]
        c3 = [(keep[6], keep[7]), (keep[9], keep[10]), (keep[12], keep[13]), (keep[15], keep[16])]
        self.conv_3 = ResidualSE(c1, c2, c3, num_block=4, groups=keep[4], kernel=(3, 3), stride=(1, 1), padding=(1, 1))
        c1 = [(keep[19], keep[20]), (keep[22], keep[23]), (keep[25], keep[26]), (keep[28], keep[29]), (keep[31], keep[32]), (keep[34], keep[35])]
        c2 = [(keep[20], keep[21]), (keep[23], keep[24]), (keep[26], keep[27]), (keep[29], keep[30]), (keep[32], keep[33]), (keep[35], keep[36])]
        c3 = [(keep[21], keep[22]), (keep[24], keep[25]), (keep[27], keep[28]), (keep[30], keep[31]), (keep[33], keep[34]), (keep[36], keep[37])]
        self.conv_4 = ResidualSE(c1, c2, c3, num_block=6, groups=keep[19], kernel=(3, 3), stride=(1, 1), padding=(1, 1))
        c1 = [(keep[40], keep[41]), (keep[43], keep[44])]; c2 = [(keep[41], keep[42]), (keep[44], keep[45])]; c3 = [(keep[42], keep[43]), (keep[45], keep[46])]
        self.conv_5 = ResidualSE(c1, c2, c3, num_block=2, groups=keep[40], kernel=(3, 3), stride=(1, 1), padding=(1, 1))

keep_dict_1_8M = [32, 32, 103, 103, 64, 13, 13, 64, 26, 26, 64, 13, 13, 64, 52, 52, 64, 231, 231, 128, 154, 154, 128, 52, 52, 128, 26, 26, 128, 52, 52, 128, 26, 26, 128, 26, 26, 128, 308, 308, 128, 26, 26, 128, 26, 26, 128, 512, 512]
keep_dict_1_8M_ = [32, 32, 103, 103, 64, 13, 13, 64, 13, 13, 64, 13, 13, 64, 13, 13, 64, 231, 231, 128, 231, 231, 128, 52, 52, 128, 26, 26, 128, 77, 77, 128, 26, 26, 128, 26, 26, 128, 308, 308, 128, 26, 26, 128, 26, 26, 128, 512, 512]

def MiniFASNetV2(embedding_size=128, conv6_kernel=(5, 5), drop_p=0.2, num_classes=3, img_channel=3):
    return MiniFASNet(keep_dict_1_8M_, embedding_size, conv6_kernel, drop_p, num_classes, img_channel)

def MiniFASNetV1SE(embedding_size=128, conv6_kernel=(5, 5), drop_p=0.75, num_classes=3, img_channel=3):
    return MiniFASNetSE(keep_dict_1_8M, embedding_size, conv6_kernel, drop_p, num_classes, img_channel)

def load_checkpoint(model_cls, weight_filename, download_url=None):
    if not os.path.exists(weight_filename):
        if download_url:
            print(f"Downloading checkpoint '{weight_filename}'...")
            urllib.request.urlretrieve(download_url, weight_filename)
        else:
            raise FileNotFoundError(f"Checkpoint '{weight_filename}' not found.")
    
    model = model_cls(embedding_size=128, conv6_kernel=(5, 5), num_classes=3).to(device)
    state_dict = torch.load(weight_filename, map_location=device)
    new_state_dict = {k[7:] if k.startswith('module.') else k: v for k, v in state_dict.items()}
    model.load_state_dict(new_state_dict)
    model.eval()
    return model

print("Loading MiniFASNet anti-spoofing model checkpoints...")
minifasnet_v2 = load_checkpoint(
    MiniFASNetV2,
    "2.7_80x80_MiniFASNetV2.pth",
    "https://github.com/minivision-ai/Silent-Face-Anti-Spoofing/raw/master/resources/anti_spoof_models/2.7_80x80_MiniFASNetV2.pth"
)
minifasnet_v1se = load_checkpoint(
    MiniFASNetV1SE,
    "4_0_0_80x80_MiniFASNetV1SE.pth",
    "https://github.com/minivision-ai/Silent-Face-Anti-Spoofing/raw/master/resources/anti_spoof_models/4_0_0_80x80_MiniFASNetV1SE.pth"
)
print("MiniFASNet anti-spoofing ensemble models ready.")

Loading MiniFASNet anti-spoofing model checkpoints...
MiniFASNet anti-spoofing ensemble models ready.


## 4. Helper Utilities for Landmark Rotation Alignment & Cropping

In [71]:
# OpenCV Face Detector & CropImage Helper with Scale Expansion
class OpenCVFaceDetector:
    def __init__(self):
        self.mode = None
        if hasattr(cv2, 'CascadeClassifier'):
            cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml' if hasattr(cv2, 'data') else 'haarcascade_frontalface_default.xml'
            self.cascade = cv2.CascadeClassifier(cascade_path)
            if not self.cascade.empty():
                self.mode = 'cascade'
        if self.mode is None and hasattr(cv2, 'FaceDetectorYN_create'):
            weights_file = "face_detection_yunet_2023mar.onnx"
            if os.path.exists(weights_file):
                self.yunet = cv2.FaceDetectorYN_create(weights_file, "", (640, 480), 0.6, 0.3, 5000)
                self.mode = 'yunet'

    def detect(self, image_bgr):
        if image_bgr is None or image_bgr.size == 0:
            return []
        h, w = image_bgr.shape[:2]
        if self.mode == 'cascade':
            gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
            faces = self.cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
            return faces if len(faces) > 0 else []
        elif self.mode == 'yunet':
            self.yunet.setInputSize((w, h))
            _, faces = self.yunet.detect(image_bgr)
            if faces is not None and len(faces) > 0:
                bboxes = []
                for f in faces:
                    x, y, bw, bh = int(f[0]), int(f[1]), int(f[2]), int(f[3])
                    bboxes.append([x, y, bw, bh])
                return np.array(bboxes)
            return []
        return []

detector_instance = OpenCVFaceDetector()

class CropImage:
    @staticmethod
    def _get_new_box(src_w, src_h, bbox, scale):
        x, y, box_w, box_h = bbox[0], bbox[1], bbox[2], bbox[3]
        scale = min((src_h - 1) / max(1, box_h), min((src_w - 1) / max(1, box_w), scale))
        new_width, new_height = box_w * scale, box_h * scale
        center_x, center_y = box_w / 2.0 + x, box_h / 2.0 + y
        left_top_x, left_top_y = center_x - new_width / 2.0, center_y - new_height / 2.0
        right_bottom_x, right_bottom_y = center_x + new_width / 2.0, center_y + new_height / 2.0
        if left_top_x < 0:
            right_bottom_x -= left_top_x; left_top_x = 0
        if left_top_y < 0:
            right_bottom_y -= left_top_y; left_top_y = 0
        if right_bottom_x > src_w - 1:
            left_top_x -= right_bottom_x - src_w + 1; right_bottom_x = src_w - 1
        if right_bottom_y > src_h - 1:
            left_top_y -= right_bottom_y - src_h + 1; right_bottom_y = src_h - 1
        return int(left_top_x), int(left_top_y), int(right_bottom_x), int(right_bottom_y)

    def crop(self, org_img, bbox, scale, out_w=80, out_h=80):
        src_h, src_w, _ = np.shape(org_img)
        left_top_x, left_top_y, right_bottom_x, right_bottom_y = self._get_new_box(src_w, src_h, bbox, scale)
        img = org_img[left_top_y: right_bottom_y + 1, left_top_x: right_bottom_x + 1]
        if img.size == 0:
            img = org_img[int(bbox[1]):int(bbox[1]+bbox[3]), int(bbox[0]):int(bbox[0]+bbox[2])]
        return cv2.resize(img, (out_w, out_h))

cropper = CropImage()
print("Detector & CropImage cropper initialized.")

Detector & CropImage cropper initialized.


[ WARN:0@2884.105] global net_impl_backend.cpp:345 setPreferableTarget Targets are not supported by the new graph engine for now


## 5. Function 1: Face Verification (`verify_face`)

Verifies if input face image matches any existing embeddings inside the `embeddings/` directory using MTCNN alignment and InceptionResNetV1 embeddings.

In [72]:
def verify_face(image_input, embeddings_dir=EMBEDDINGS_DIR, threshold=0.65):
    """
    Function 1: Face Verification
    - Scans directory 'embeddings' for registered embeddings (.npy files).
    - Extracts candidate face embedding using MTCNN & InceptionResNetV1.
    - Computes Cosine Similarity against all registered face embeddings.
    
    Input:
        image_input: BGR numpy array OR list of BGR numpy arrays (uses primary frame).
    Returns:
        (is_matched: bool, identity_name: str, max_similarity: float)
    """
    if isinstance(image_input, list):
        image_bgr = image_input[-1] if len(image_input) > 0 else None
    else:
        image_bgr = image_input

    if image_bgr is None or image_bgr.size == 0:
        print("[Face Verification] Invalid image input.")
        return False, "Invalid Input", 0.0

    frame_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(frame_rgb).convert("RGB")

    # Detect & Align face using MTCNN
    face = mtcnn(pil_img)
    if face is None:
        print("[Face Verification] No face detected by MTCNN.")
        return False, "No Face Detected", 0.0

    # Extract 512-dim embedding using InceptionResNetV1
    face_tensor = face.unsqueeze(0).to(device)
    with torch.no_grad():
        candidate_emb = resnet(face_tensor).cpu().numpy()

    if not os.path.exists(embeddings_dir):
        os.makedirs(embeddings_dir, exist_ok=True)

    emb_files = [f for f in os.listdir(embeddings_dir) if f.endswith('.npy')]
    if not emb_files:
        print(f"[Face Verification] No registered embeddings found in '{embeddings_dir}'.")
        return False, "No Registered Embeddings", 0.0

    best_match_identity = "Unknown"
    max_similarity = -1.0
    c_emb_tensor = torch.tensor(candidate_emb)

    for ef in emb_files:
        path = os.path.join(embeddings_dir, ef)
        try:
            reg_emb = np.load(path)
            r_emb_tensor = torch.tensor(reg_emb)
            sim = F.cosine_similarity(c_emb_tensor, r_emb_tensor).item()
            identity_name = os.path.splitext(ef)[0]
            if sim > max_similarity:
                max_similarity = sim
                best_match_identity = identity_name
        except Exception as e:
            print(f"[Face Verification] Warning: Error reading {path}: {e}")

    is_matched = (max_similarity >= threshold)
    print(f"[Face Verification] Best Match: '{best_match_identity}' | Cosine Similarity: {max_similarity:.4f} | Threshold: {threshold} | Matched: {is_matched}")
    
    return is_matched, best_match_identity, max_similarity

## 6. Function 2: Anti-Spoofing Detection (`anti_spoofing_detection`)

Enhanced passive anti-spoofing engine with **Facial Landmark Rotation Normalization**, **Close-Distance Anomaly Guard**, and **Multi-Frame Temporal Aggregation**.

In [73]:
CLASS_MAPPING = {
    0: "SPOOF (Print Attack)",
    1: "REAL (Genuine Face)",
    2: "SPOOF (Replay Attack)"
}

def predict_single_frame_spoof(image_bgr):
    """
    Evaluates anti-spoofing probabilities for a single frame with:
    1. MTCNN landmark detection & rotation alignment (normalizing roll angle theta to upright 0 deg).
    2. Close-distance ratio check (detects phone held too close to lens).
    3. Multi-scale MiniFASNet ensemble evaluation.
    """
    if image_bgr is None or image_bgr.size == 0:
        return None

    h, w = image_bgr.shape[:2]
    frame_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(frame_rgb)

    # Detect face & 5-point facial landmarks using MTCNN
    boxes, probs, points = mtcnn.detect(pil_img, landmarks=True)

    bbox = None
    angle = 0.0
    aligned_img = image_bgr.copy()

    if boxes is not None and len(boxes) > 0:
        b = boxes[0]
        x1, y1, x2, y2 = int(b[0]), int(b[1]), int(b[2]), int(b[3])
        bw, bh = x2 - x1, y2 - y1
        bbox = [max(0, x1), max(0, y1), bw, bh]

        # Calculate rotation angle using eye landmarks
        if points is not None and len(points) > 0:
            landmarks = points[0]
            left_eye, right_eye = landmarks[0], landmarks[1]
            dy = right_eye[1] - left_eye[1]
            dx = right_eye[0] - left_eye[0]
            angle = float(np.degrees(np.arctan2(dy, dx)))

            # Extreme unnatural tilt check (>30 deg tilt typical of rotated phone attack)
            if abs(angle) > 30.0:
                # Return high replay spoof probability for extreme tilt
                return np.array([0.05, 0.05, 0.90])

            # Perform affine rotation alignment to upright 0 deg if tilted
            if abs(angle) > 3.0:
                eye_center = (float((left_eye[0] + right_eye[0]) / 2.0), float((left_eye[1] + right_eye[1]) / 2.0))
                M = cv2.getRotationMatrix2D(eye_center, angle, 1.0)
                aligned_img = cv2.warpAffine(image_bgr, M, (w, h), flags=cv2.INTER_CUBIC)

    # Fallback to OpenCV detector if MTCNN missed
    if bbox is None:
        faces = detector_instance.detect(image_bgr)
        if len(faces) > 0:
            bbox = faces[0]
        else:
            return None

    # Distance & Close-range check: Face filling >38% frame area or height >65% frame height
    bw, bh = bbox[2], bbox[3]
    area_ratio = (bw * bh) / float(w * h)
    if area_ratio > 0.38 or bh > 0.65 * h:
        # Phone screen held too close to camera lens! Flag as replay attack
        return np.array([0.05, 0.05, 0.90])

    # Multi-scale crops from normalized upright image
    crop_27 = cropper.crop(aligned_img, bbox, scale=2.7, out_w=80, out_h=80)
    crop_40 = cropper.crop(aligned_img, bbox, scale=4.0, out_w=80, out_h=80)

    t_27 = torch.from_numpy(crop_27.transpose((2, 0, 1))).float().unsqueeze(0).to(device)
    t_40 = torch.from_numpy(crop_40.transpose((2, 0, 1))).float().unsqueeze(0).to(device)

    minifasnet_v2.eval()
    minifasnet_v1se.eval()

    with torch.no_grad():
        prob_v2 = F.softmax(minifasnet_v2(t_27), dim=1).cpu().numpy()
        prob_v1se = F.softmax(minifasnet_v1se(t_40), dim=1).cpu().numpy()

    combined_prob = (prob_v2 + prob_v1se) / 2.0  # Normalized [0.0, 1.0]
    return combined_prob[0]


def anti_spoofing_detection(image_input, threshold=0.60, max_replay_thresh=0.30):
    """
    Function 2: Anti-Spoofing Detection with Multi-Frame Temporal Aggregation & Robustness Guards
    - Accepts a single frame OR list of frames.
    - Performs rotation warping, distance validation, and ensemble prediction.
    - Enforces real_score >= threshold and replay_score < max_replay_thresh.
    
    Returns:
        (is_real: bool, class_label: str, avg_real_score: float)
    """
    if isinstance(image_input, list):
        frames = [f for f in image_input if f is not None and f.size > 0]
    else:
        frames = [image_input] if (image_input is not None and image_input.size > 0) else []

    if not frames:
        print("[Anti-Spoofing] Invalid image input.")
        return False, "Invalid Input", 0.0

    probs_list = []
    for frame in frames:
        prob = predict_single_frame_spoof(frame)
        if prob is not None:
            probs_list.append(prob)

    if not probs_list:
        print("[Anti-Spoofing] No face detected across frame sample.")
        return False, "No Face Detected", 0.0

    # Temporal average of probabilities over sampled frames
    avg_prob = np.mean(probs_list, axis=0)
    print_score = float(avg_prob[0])
    real_score = float(avg_prob[1])
    replay_score = float(avg_prob[2])

    pred_class = int(np.argmax(avg_prob))
    class_label = CLASS_MAPPING.get(pred_class, "SPOOF")

    # Strict Liveness Decision Guard
    is_real = (pred_class == 1 and real_score >= threshold and replay_score < max_replay_thresh)

    print(f"[Anti-Spoofing] Sampled Frames: {len(probs_list)} | Label: '{class_label}'")
    print(f"[Anti-Spoofing] Scores -> Real: {real_score:.4f} | Print: {print_score:.4f} | Replay: {replay_score:.4f}")
    print(f"[Anti-Spoofing] Real Threshold: {threshold} | Max Replay Allowed: {max_replay_thresh} | Result Is Real: {is_real}")
    
    return is_real, class_label, real_score

## 7. Camera Capture & Main Verification Function (`overall_verification`)

1. **`capture_picture()`**: Opens live camera window with **Real-Time Live Anti-Spoofing & Distance/Tilt Warnings**. Captures a 7-frame temporal burst when user presses `'C'` (or `'c'`).
2. **`overall_verification()`**: Runs both `verify_face()` and `anti_spoofing_detection()`. Returns **`True` if and only if both return `True`**, else **`False`**.

In [74]:
def capture_picture(window_size=7):
    """
    Opens live webcam feed with live anti-spoofing and distance/rotation warning overlays.
    When user presses 'C', captures a 7-frame temporal sequence for robust verification.
    Press 'Q' to cancel capture.
    """
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: Unable to access camera device (VideoCapture(0)).")
        return None

    print("\n--- Webcam Active ---")
    print("Instruction: Press 'C' to Click / Capture Picture (7-frame temporal burst).")
    print("Instruction: Press 'Q' to Quit / Cancel.")

    recent_frames = deque(maxlen=window_size)
    real_history = deque(maxlen=10)
    captured_sequence = None

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Error reading webcam frame.")
            break

        # Flip horizontally for mirror preview
        display_frame = cv2.flip(frame, 1)
        recent_frames.append(display_frame.copy())
        h, w = display_frame.shape[:2]

        # Live liveness estimation and distance/rotation overlay
        prob = predict_single_frame_spoof(display_frame)
        if prob is not None:
            real_history.append(float(prob[1]))
            avg_real = float(np.mean(real_history))
            live_status = "REAL" if avg_real >= 0.50 else "SPOOF"
            color = (0, 255, 0) if live_status == "REAL" else (0, 0, 255)
            cv2.putText(display_frame, f"LIVE LIVENESS: {live_status} ({avg_real*100:.1f}%)",
                        (20, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)

        # On-screen header overlay
        cv2.putText(display_frame, "PRESS 'C' TO CAPTURE PICTURE | PRESS 'Q' TO QUIT",
                    (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

        cv2.imshow("Authentication System - Press C to Capture", display_frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('c') or key == ord('C'):
            print("Capture triggered! Sampling 7-frame temporal sequence...")
            burst_frames = list(recent_frames)
            while len(burst_frames) < window_size:
                r, f = cap.read()
                if r:
                    burst_frames.append(cv2.flip(f, 1))
                else:
                    break
            captured_sequence = burst_frames
            print(f"Captured {len(captured_sequence)} frames successfully!")
            break
        elif key == ord('q') or key == ord('Q'):
            print("Camera capture cancelled by user.")
            break

    cap.release()
    cv2.destroyAllWindows()
    return captured_sequence


def overall_verification(image_input=None, similarity_threshold=0.65, spoof_threshold=0.60):
    """
    Main Verification Function:
    1. Opens camera (if image_input is None) and captures 7-frame sequence when user presses 'C'.
    2. Runs Function 1: Face Verification (MTCNN + InceptionResNetV1).
    3. Runs Function 2: Anti-Spoofing Detection (MiniFASNet Ensemble with Landmark Rotation & Distance Guards).
    
    Returns:
        True ONLY if both verify_face and anti_spoofing_detection return True.
        Else False.
    """
    print("\n=======================================================")
    print("       STARTING OVERALL FACE AUTHENTICATION          ")
    print("=======================================================")

    # Step 0: Get Image/Frames (Camera Capture or Provided Image)
    if image_input is None:
        frames = capture_picture()
    else:
        frames = image_input

    if frames is None:
        print("[AUTHENTICATION FAILED] No frames captured.")
        return False

    # Step 1: Run Face Verification
    print("\n[Step 1] Running Face Verification...")
    face_match_pass, identity, sim_score = verify_face(
        frames,
        embeddings_dir=EMBEDDINGS_DIR,
        threshold=similarity_threshold
    )

    # Step 2: Run Anti-Spoofing Detection
    print("\n[Step 2] Running Anti-Spoofing Detection...")
    liveness_pass, spoof_label, real_prob = anti_spoofing_detection(
        frames,
        threshold=spoof_threshold
    )

    # Final Decision Evaluation
    overall_status = face_match_pass and liveness_pass

    print("\n-------------------------------------------------------")
    print("                 VERIFICATION SUMMARY                 ")
    print("-------------------------------------------------------")
    print(f" 1. Face Verification Status : {'PASSED [OK]' if face_match_pass else 'FAILED [X]'}")
    print(f"    - Matched Identity       : {identity}")
    print(f"    - Cosine Similarity      : {sim_score:.4f} (Threshold: {similarity_threshold})")
    print(f" 2. Anti-Spoofing Status    : {'PASSED [OK]' if liveness_pass else 'FAILED [X]'}")
    print(f"    - Prediction Label       : {spoof_label}")
    print(f"    - Real Probability       : {real_prob:.4f} (Threshold: {spoof_threshold})")
    print("-------------------------------------------------------")
    
    if overall_status:
        print(" >>> FINAL RESULT: TRUE (ACCESS GRANTED) <<<")
    else:
        print(" >>> FINAL RESULT: FALSE (ACCESS DENIED) <<<")
    print("=======================================================\n")

    return overall_status

## 8. User Registration Utility (`register_new_face`)

Utility function to capture face and store its embedding into `embeddings/<identity_name>.npy`.

In [ ]:
def register_new_face(name=None, image_bgr=None):
    """
    Helper function to register a new user's face embedding into 'embeddings/<name>.npy'.
    """
    if name is None:
        name = input("Enter identity name for registration (e.g., 'john_doe'): ").strip()
        if not name:
            print("Invalid name. Registration cancelled.")
            return False

    if image_bgr is None:
        print(f"Capturing face for '{name}' via webcam...")
        sequence = capture_picture()
        image_bgr = sequence[-1] if (sequence and len(sequence) > 0) else None

    if image_bgr is None:
        print("No picture captured for registration.")
        return False

    frame_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(frame_rgb).convert("RGB")

    face = mtcnn(pil_img)
    if face is None:
        print("No face detected by MTCNN. Registration failed.")
        return False

    face_tensor = face.unsqueeze(0).to(device)
    with torch.no_grad():
        embedding = resnet(face_tensor).cpu().numpy()

    save_path = os.path.join(EMBEDDINGS_DIR, f"{name}.npy")
    np.save(save_path, embedding)
    print(f"Successfully registered embedding for '{name}' -> Saved to '{save_path}'")
    return True

# register_new_face()

## 9. Run Overall Verification System

In [78]:
# Run authentication system
# Note: When executed in an interactive environment with webcam, it will open the camera feed.
result = overall_verification()
print(f"overall_verification() return value: {result}")


       STARTING OVERALL FACE AUTHENTICATION          

--- Webcam Active ---
Instruction: Press 'C' to Click / Capture Picture (7-frame temporal burst).
Instruction: Press 'Q' to Quit / Cancel.
Capture triggered! Sampling 7-frame temporal sequence...
Captured 7 frames successfully!

[Step 1] Running Face Verification...
[Face Verification] Best Match: 'antik' | Cosine Similarity: 0.8269 | Threshold: 0.65 | Matched: True

[Step 2] Running Anti-Spoofing Detection...
[Anti-Spoofing] Sampled Frames: 7 | Label: 'SPOOF (Print Attack)'
[Anti-Spoofing] Scores -> Real: 0.2647 | Print: 0.6617 | Replay: 0.0736
[Anti-Spoofing] Real Threshold: 0.6 | Max Replay Allowed: 0.3 | Result Is Real: False

-------------------------------------------------------
                 VERIFICATION SUMMARY                 
-------------------------------------------------------
 1. Face Verification Status : PASSED [OK]
    - Matched Identity       : antik
    - Cosine Similarity      : 0.8269 (Threshold: 0.65)
 2